In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

print("Pytorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Pytorch version: 2.11.0+cpu
CUDA available: False
Using device: cpu


## Tensors - the beginning

In [6]:
x = torch.tensor([1.0, 2.0, 3.0])
print(x)
print(type(x))
print(x.shape)
print(x.dtype)

tensor([1., 2., 3.])
<class 'torch.Tensor'>
torch.Size([3])
torch.float32


In [7]:
zeros = torch.zeros(2,3)
ones = torch.ones(2,3)
rand = torch.randn(2,3)

print("zeros:\n", zeros)
print("ones:\n", ones)
print("rand:\n", rand)

zeros:
 tensor([[0., 0., 0.],
        [0., 0., 0.]])
ones:
 tensor([[1., 1., 1.],
        [1., 1., 1.]])
rand:
 tensor([[-1.1481, -1.2090,  1.8418],
        [ 0.4000, -1.0335,  1.3687]])


In [10]:
a = torch.tensor([[1,2,3],
                 [4,5,6]])
print(a)
print("shape:", a.shape)
print("ndim:", a.ndim)

tensor([[1, 2, 3],
        [4, 5, 6]])
shape: torch.Size([2, 3])
ndim: 2


## Basic Tensor Operations

In [11]:
x = torch.tensor([1.0,2.0, 3.0])
y = torch.tensor([4.0, 5.0, 6.0])

print("x + y =", x + y)
print("x - y=", x -y)
print("x * y=", x * y)
print("x /y =", x/y)


x + y = tensor([5., 7., 9.])
x - y= tensor([-3., -3., -3.])
x * y= tensor([ 4., 10., 18.])
x /y = tensor([0.2500, 0.4000, 0.5000])


In [12]:
A = torch.tensor([[1.0, 2.0],
                 [3.0, 4.0]])
B = torch.tensor([[5.0, 6.0],
                  [7.0, 8.0]])

print("A @ B =\n", A @ B)


A @ B =
 tensor([[19., 22.],
        [43., 50.]])


## Reshaping Tensors

In [13]:
x = torch.arange(12)
print("x:", x)
print("shape:", x.shape)

x2 = x.reshape(3,4)
print("\nreshaped to 3x4:\n", x2)
print("shape:", x2.shape)


x: tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])
shape: torch.Size([12])

reshaped to 3x4:
 tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
shape: torch.Size([3, 4])


In [14]:
flat = x2.reshape(-1)
print(flat)
print(flat.shape)

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])
torch.Size([12])


## Indexing & Slicing

In [19]:
x = torch.tensor([[10,20,30],
                  [40,50,60]])
print("first row:", x[0])
print("second row:", x[1])
print("element at row 0 col 2:", x[0,2])
print("first two columns:\n", x[:, :2])

first row: tensor([10, 20, 30])
second row: tensor([40, 50, 60])
element at row 0 col 2: tensor(30)
first two columns:
 tensor([[10, 20],
        [40, 50]])


In [20]:
# moving tensors to gpu
x = torch.randn(2,2)
x = x.to(device)
print(x)
print(x.device)

tensor([[-1.0885, -1.8901],
        [-0.4716, -0.8458]])
cpu


## Autograd: the magic that trains neural networks

In [22]:
x = torch.tensor(3.0, requires_grad = True)
y = x**2 + 2*x + 1

print("x:", x)
print("y:", y)



x: tensor(3., requires_grad=True)
y: tensor(16., grad_fn=<AddBackward0>)


In [24]:
# requires_grad=True tells PyTorch:
# track operations on this tensor
# y is computed from x

y.backward()
print("dy/dx=", x.grad)


dy/dx= tensor(8.)


## Gradients with vectors

In [27]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = (x ** 2).sum()
print("y:", y)
y.backward()
print("x.grad:", x.grad)
# If ( y = x_1^2 + x_2^2 + x_3^2 ), gradient is: [2*x1, 2*x2, 2*x3]

y: tensor(14., grad_fn=<SumBackward0>)
x.grad: tensor([2., 4., 6.])


## Very important gradient habit: zeroing gradients

In [28]:
x = torch.tensor(2.0, requires_grad=True)

y1 = x ** 2
y1.backward()
print("After first backward:", x.grad)

y2 = 3 * x
y2.backward()
print("After second backward:", x.grad)

After first backward: tensor(4.)
After second backward: tensor(7.)


In [30]:
x = torch.tensor([1.0], requires_grad=True)
optimizer = optim.SGD([x], lr=0.01)

loss = (x ** 2).sum()

optimizer.zero_grad()
loss.backward()
optimizer.step()

print("Optimizer and Loss are now defined and used correctly.")
print("x after step:", x)
print("x.grad after zero_grad:", x.grad)

Optimizer and Loss are now defined and used correctly.
x after step: tensor([0.9800], requires_grad=True)
x.grad after zero_grad: tensor([2.])


## Build your first model with nn.Module

In [33]:
class TinyNet(nn.Module):
  def __init__(self):
    super().__init__()
    self.layer1 = nn.Linear(2,4)
    self.layer2 = nn.Linear(4,1)

  def forward(self, x):
    x = self.layer1(x)
    x = torch.relu(x)
    x = self.layer2(x)
    return x




In [34]:
model = TinyNet()
print(model)

TinyNet(
  (layer1): Linear(in_features=2, out_features=4, bias=True)
  (layer2): Linear(in_features=4, out_features=1, bias=True)
)


In [35]:
x = torch.tensor([[1.0, 2.0],
                  [3.0, 4.0]])

y = model(x)

print("input shape:", x.shape)
print("output shape:", y.shape)
print(y)

input shape: torch.Size([2, 2])
output shape: torch.Size([2, 1])
tensor([[-0.3707],
        [-0.3547]], grad_fn=<AddmmBackward0>)


## See the parameters

In [36]:
for name, param in model.named_parameters():
  print(name, param.shape)

layer1.weight torch.Size([4, 2])
layer1.bias torch.Size([4])
layer2.weight torch.Size([1, 4])
layer2.bias torch.Size([1])


## Tiny Training example

In [37]:
torch.manual_seed(42)
X = torch.linspace(-5,5,100).reshape(-1,1)
Y = 2*X +1

print(X[:5])
print(Y[:5])
print(X.shape, Y.shape)

tensor([[-5.0000],
        [-4.8990],
        [-4.7980],
        [-4.6970],
        [-4.5960]])
tensor([[-9.0000],
        [-8.7980],
        [-8.5960],
        [-8.3939],
        [-8.1919]])
torch.Size([100, 1]) torch.Size([100, 1])


100 points from -5 to 5

each input has shape (1,)

targets follow the line 2x + 1

In [38]:
model = nn.Linear(1,1)
print(model.weight, model.bias)

Parameter containing:
tensor([[0.7645]], requires_grad=True) Parameter containing:
tensor([0.8300], requires_grad=True)


## defining loss and optimizer

In [43]:
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

## Training loop

In [44]:
num_epochs = 200

for epoch in range(num_epochs):
  # Forward pass
  predictions = model(X)
  loss = criterion(predictions, Y)

  # Backward pass
  optimizer.zero_grad()
  loss.backward()

  # Update parameters
  optimizer.step()

  if epoch % 20 ==0:
    print(f"Epoch {epoch:03d} | Loss: {loss.item():.6f}")



Epoch 000 | Loss: 0.000000
Epoch 020 | Loss: 0.000000
Epoch 040 | Loss: 0.000000
Epoch 060 | Loss: 0.000000
Epoch 080 | Loss: 0.000000
Epoch 100 | Loss: 0.000000
Epoch 120 | Loss: 0.000000
Epoch 140 | Loss: 0.000000
Epoch 160 | Loss: 0.000000
Epoch 180 | Loss: 0.000000


In [45]:
print("Learned weight:", model.weight.item())
print("Learned bias:", model.bias.item())

Learned weight: 1.999999761581421
Learned bias: 0.9999985694885254


## Make predictions

In [46]:
text_x = torch.tensor([[10.0], [-2.5], [0.0]])
test_y = model(text_x)
print(test_y)

tensor([[21.0000],
        [-4.0000],
        [ 1.0000]], grad_fn=<AddmmBackward0>)


## Move model and data to device

In [47]:
model = nn.Linear(1,1).to(device)
X_device = X.to(device)
Y_device = Y.to(device)

criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

for epoch in range(100):
  prediction = model(X_device)
  loss = criterion(prediction, Y_device)

  optimizer.zero_grad()
  loss.backward()
  optimizer.step()

print("weight:", model.weight.item())
print("bias:", model.bias.item())

weight: 1.999999761581421
bias: 0.9892063140869141


## Save and load models

In [48]:
torch.save(model.state_dict(), "linear_model.pth")
print("Model saved")

Model saved


## Load into a new model

In [49]:
loaded_model = nn.Linear(1,1).to(device)
loaded_model.load_state_dict(torch.load("linear_model.pth", map_location=device))
loaded_model.eval()

print("Loaded weight:", loaded_model.weight.item())
print("Loaaded bias:", loaded_model.bias.item())

Loaded weight: 1.999999761581421
Loaaded bias: 0.9892063140869141


In [50]:
# Make a reusable training function
def train_linear_model(model, X, Y, epochs=100, lr=0.01, device="cpu"):
    model = model.to(device)
    X = X.to(device)
    Y = Y.to(device)

    criterion = nn.MSELoss()
    optimizer = optim.SGD(model.parameters(), lr=lr)

    for epoch in range(epochs):
        preds = model(X)
        loss = criterion(preds, Y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if epoch % 20 == 0:
            print(f"Epoch {epoch:03d} | Loss: {loss.item():.6f}")

    return model

In [51]:
model2 = nn.Linear(1, 1)
model2 = train_linear_model(model2, X, Y, epochs=100, lr=0.01, device=device)

print("final weight:", model2.weight.item())
print("final bias:", model2.bias.item())

Epoch 000 | Loss: 52.748554
Epoch 020 | Loss: 0.106343
Epoch 040 | Loss: 0.033855
Epoch 060 | Loss: 0.015081
Epoch 080 | Loss: 0.006722
final weight: 1.999999761581421
final bias: 0.9452656507492065
